# Statlig SVV-vegfinansiering per landsdel 2005–2039

Analyse basert på arket **Prosjekter_SVV_2005-2026** i
`Samferdselprosjekter_pr_landsdel_over_tid_public.xlsx`.

Alle beløp i mill. 2026-kr. Statlig finansiering = `Kostnad stat 2026kr`-kolonnen.

**Relevans:** Underlag for kritikk av Hordfast-argumentasjonen til Initiativ Vest.

## 1. Biblioteker og konfigurasjon

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.axisbelow': True,
})

FARGER = {
    'ØST':  '#185FA5',
    'VEST': '#D85A30',
    'NORD': '#0F6E56',
    'MIDT': '#7F77DD',
    'SØR':  '#888780',
}
RNAVN = {
    'ØST':  'Østlandet',
    'VEST': 'Vestlandet',
    'NORD': 'Nord-Norge',
    'MIDT': 'Trøndelag',
    'SØR':  'Sørlandet',
}
FOLK = {
    'ØST':  2_870_000,
    'VEST': 1_420_000,
    'NORD':   480_000,
    'MIDT':   480_000,
    'SØR':    320_000,
}
TOTAL_FOLK = sum(FOLK.values())
BEF = {ld: FOLK[ld] / TOTAL_FOLK * 100 for ld in FOLK}
REGIONS = ['ØST', 'VEST', 'NORD', 'MIDT', 'SØR']

## 2. Last og rens data

In [ ]:
XLSX = 'Samferdselprosjekter_pr_landsdel_over_tid_public.xlsx'

df = pd.read_excel(XLSX, sheet_name='Prosjekter_SVV_2005-2026', header=4)
df = df.dropna(subset=['Prosjekt']).copy()
df['aar']   = pd.to_numeric(df['Trafikkåpning_bruk'], errors='coerce')
df['stat']  = pd.to_numeric(df['Kostnad stat 2026kr'], errors='coerce').fillna(0)
df['total'] = pd.to_numeric(df['Prognose/endelig sluttkostnad (indeksert)'], errors='coerce').fillna(0)
df['andel'] = pd.to_numeric(df['Statlig andel (sjekket)'], errors='coerce').fillna(0)
df = df[df['aar'] >= 2005].copy()

print(f'Prosjekter: {len(df)}   |   Sum statlig: {df["stat"].sum()/1000:.1f} mrd   |'
      f'   Periode: {int(df["aar"].min())}–{int(df["aar"].max())}')
df[['Prosjekt','Landsdel','aar','total','andel','stat']].head()

## 3. Aggreger per 3-årsperiode

In [ ]:
PERIODER = [
    (2005, 2007, '2005–07'),
    (2008, 2010, '2008–10'),
    (2011, 2013, '2011–13'),
    (2014, 2016, '2014–16'),
    (2017, 2019, '2017–19'),
    (2020, 2022, '2020–22'),
    (2023, 2025, '2023–25'),
    (2026, 2039, '2026+'),
]
N_AAR = {lbl: (e - s + 1) if e <= 2025 else 10 for s, e, lbl in PERIODER}

rows = []
for s, e, lbl in PERIODER:
    for ld in REGIONS:
        v = df[(df['Landsdel'] == ld) & (df['aar'] >= s) & (df['aar'] <= e)]['stat'].sum()
        rows.append({'periode': lbl, 'landsdel': ld, 'stat': v})
period_df = pd.DataFrame(rows)

# Pivoter
pivot = period_df.pivot(index='periode', columns='landsdel', values='stat').fillna(0)
pivot = pivot.loc[[lbl for _, _, lbl in PERIODER], REGIONS]
pivot['total'] = pivot.sum(axis=1)

# Vestlandets andel
pivot['vest_pct'] = pivot['VEST'] / pivot['total'] * 100

# Kr per innbygger per år
for ld in REGIONS:
    pivot[f'kr_{ld}'] = pivot.apply(
        lambda r: r[ld] * 1e6 / FOLK[ld] / N_AAR[r.name], axis=1
    )

pivot.round(1)

## 4. Statlig SVV per periode (stablet søylediagram)

Vestlandet dominerer **2026+** med 52,9 mrd – 50,8 % av total i perioden.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(pivot))
bottom = np.zeros(len(pivot))

for ld in REGIONS:
    vals = pivot[ld].values / 1000
    ax.bar(x, vals, bottom=bottom, label=RNAVN[ld], color=FARGER[ld], width=0.7)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel('Mrd 2026-kr')
ax.set_title('Statlig SVV-riksvegfinansiering per 3-årsperiode', fontsize=13, fontweight='500')
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Vestlandets andel per periode

Stiplede linje viser befolkningsandelen på 25,5 %. Vestlandet er under bef.andelen i alle perioder
2005–2019, og langt over i 2026+.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.fill_between(pivot.index, pivot['vest_pct'], 25.5,
                where=pivot['vest_pct'] > 25.5,
                alpha=0.25, color=FARGER['VEST'], label='Over befolkning')
ax.fill_between(pivot.index, pivot['vest_pct'], 25.5,
                where=pivot['vest_pct'] <= 25.5,
                alpha=0.18, color='#888780', label='Under befolkning')

ax.plot(pivot.index, pivot['vest_pct'], 'o-',
        color=FARGER['VEST'], lw=2.2, ms=6, label='Vestlandet andel')
ax.axhline(25.5, color='#888780', lw=1.3, ls='--', label='Befolkningsandel 25,5 %')

for i, (idx, row) in enumerate(pivot.iterrows()):
    ax.annotate(f"{row['vest_pct']:.1f} %",
                xy=(i, row['vest_pct']),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=9.5, color=FARGER['VEST'])

ax.set_ylim(0, 75)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v)} %'))
ax.set_ylabel('Andel av total statlig SVV (%)')
ax.set_title('Vestlandets andel av statlig SVV-finansiering', fontsize=13, fontweight='500')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 6. Statlig SVV kr per innbygger per år

Normalisert for befolkningsstørrelse. Nord-Norge leder i 2017–19 (9 660 kr/innb/år),
Vestlandet tar ledelsen i 2026+ (3 727 kr/innb/år).

In [ ]:
n = len(pivot)
x = np.arange(n)
w = 0.16
offsets = np.linspace(-(len(REGIONS)-1)/2, (len(REGIONS)-1)/2, len(REGIONS)) * w

fig, ax = plt.subplots(figsize=(12, 5.5))
for i, ld in enumerate(REGIONS):
    vals = pivot[f'kr_{ld}'].values
    ax.bar(x + offsets[i], vals, w * 0.95,
           label=RNAVN[ld], color=FARGER[ld])

ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v/1000)}k'))
ax.set_ylabel('Kr per innbygger per år')
ax.set_title('Statlig SVV-finansiering kr per innbygger per år', fontsize=13, fontweight='500')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Vektet statlig andel på Vestlandets prosjekter per periode

Lav andel = høy bompengebelastning for trafikantene.

In [ ]:
vest_andel = []
vest_periode = []
for s, e, lbl in PERIODER:
    sub = df[(df['Landsdel'] == 'VEST') & (df['aar'] >= s) & (df['aar'] <= e)]
    if len(sub) > 0 and sub['total'].sum() > 0:
        vektet = (sub['andel'] * sub['total']).sum() / sub['total'].sum() * 100
        vest_andel.append(round(vektet, 1))
        vest_periode.append(lbl)
    else:
        vest_andel.append(None)
        vest_periode.append(lbl)

fig, ax = plt.subplots(figsize=(11, 4))
bar_colors = [FARGER['VEST'] if (v is not None and v >= 50) else '#D85A3066'
              if v is not None else 'none' for v in vest_andel]
bars = ax.bar(range(len(vest_periode)),
              [v if v is not None else 0 for v in vest_andel],
              color=bar_colors, width=0.65)

for i, v in enumerate(vest_andel):
    if v is not None and v > 0:
        ax.text(i, v + 1.5, f'{v:.0f} %', ha='center', fontsize=10, color='#444')

ax.axhline(50, color='#888780', lw=1.2, ls='--', alpha=0.7, label='50 % (halvparten statlig)')
ax.set_xticks(range(len(vest_periode)))
ax.set_xticklabels(vest_periode)
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v)} %'))
ax.set_ylabel('Statlig andel (vektet gjennomsnitt)')
ax.set_title('Vektet statlig finansieringsandel – Vestlandets SVV-prosjekter', fontsize=13, fontweight='500')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 8. Alle Vestlands-prosjekter sortert etter statlig andel

Fjordkrysninger og bypakkeprosjekter har lavest statlig andel (bompenger);
rassikringsprosjekter og tunneloppgraderinger har 100 % statlig.

In [ ]:
vest = df[df['Landsdel'] == 'VEST'].copy()
vest = vest.sort_values('andel', ascending=True)
vest['bom'] = vest['total'] - vest['stat']
vest['etikett'] = vest['Prosjekt'].str[:45]

fig, ax = plt.subplots(figsize=(12, max(7, len(vest) * 0.4)))
y = np.arange(len(vest))
h = 0.6

ax.barh(y, vest['stat'].values / 1000,   h, color=FARGER['VEST'], label='Statlig')
ax.barh(y, vest['bom'].values  / 1000,   h,
        left=vest['stat'].values / 1000, color='#D85A3044', label='Bompenger',
        edgecolor=FARGER['VEST'], linewidth=0.4)

for i, (_, row) in enumerate(vest.iterrows()):
    pct = row['andel'] * 100
    total_mrd = row['total'] / 1000
    ax.text(total_mrd + 0.15, i, f'{pct:.0f} % stat', va='center', fontsize=8.5)

ax.set_yticks(y)
ax.set_yticklabels(vest['etikett'], fontsize=9)
ax.set_xlabel('Mrd 2026-kr')
ax.set_title('Vestlandets SVV-prosjekter 2005-2039 (sortert etter statlig andel, lavest oeverst)',
             fontsize=13, fontweight='500')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 9. Kumulativ oppsummering: ferdige 2005–2025 vs under bygging 2026+

In [ ]:
ferdig = df[df['aar'] <= 2025].groupby('Landsdel')['stat'].sum().reindex(REGIONS, fill_value=0)
bygges = df[df['aar'] >  2025].groupby('Landsdel')['stat'].sum().reindex(REGIONS, fill_value=0)
totalt = ferdig + bygges

tf, tb, tt = ferdig.sum(), bygges.sum(), totalt.sum()

summary = pd.DataFrame({
    'Befolkning (%)':          [round(BEF[ld], 1) for ld in REGIONS],
    'Ferdig 2005–2025 (mrd)':  [round(ferdig[ld]/1000, 1) for ld in REGIONS],
    'F-andel (%)':             [round(ferdig[ld]/tf*100, 1) for ld in REGIONS],
    'F-avvik (pp)':            [round(ferdig[ld]/tf*100 - BEF[ld], 1) for ld in REGIONS],
    'Bygges 2026+ (mrd)':      [round(bygges[ld]/1000, 1) for ld in REGIONS],
    'B-andel (%)':             [round(bygges[ld]/tb*100, 1) for ld in REGIONS],
    'B-avvik (pp)':            [round(bygges[ld]/tb*100 - BEF[ld], 1) for ld in REGIONS],
    'Total (mrd)':             [round(totalt[ld]/1000, 1) for ld in REGIONS],
    'T-andel (%)':             [round(totalt[ld]/tt*100, 1) for ld in REGIONS],
    'T-avvik (pp)':            [round(totalt[ld]/tt*100 - BEF[ld], 1) for ld in REGIONS],
}, index=[RNAVN[ld] for ld in REGIONS])

summary

## 10. Kumulativt: andel vs befolkningsandel

Stiplet linje = befolkningsandel per landsdel.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=False)
titler = ['Ferdig 2005–2025', 'Under bygging 2026+', 'Total 2005–2039']
andeler = [
    [round(ferdig[ld]/tf*100, 1) for ld in REGIONS],
    [round(bygges[ld]/tb*100, 1) for ld in REGIONS],
    [round(totalt[ld]/tt*100, 1) for ld in REGIONS],
]

for ax, tittel, ands in zip(axes, titler, andeler):
    bars = ax.bar([RNAVN[ld] for ld in REGIONS], ands,
                  color=[FARGER[ld] for ld in REGIONS], width=0.6)
    for bar, a, ld in zip(bars, ands, REGIONS):
        ax.axhline(BEF[ld], xmin=(bar.get_x()-(-0.5))/(len(REGIONS)),
                   xmax=(bar.get_x()+bar.get_width()-(-0.5))/(len(REGIONS)),
                   color='#333', lw=1.5, ls='--')
    ax.set_title(tittel, fontsize=11, fontweight='500')
    ax.set_xticklabels([RNAVN[ld] for ld in REGIONS], rotation=25, ha='right', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v)} %'))
    ax.set_ylabel('Andel av total statlig SVV (%)')

fig.suptitle('Andel av statlig SVV – stiplet linje viser befolkningsandel',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()

## 11. Konklusjoner for Hordfast/Initiativ Vest-debatten

### Hva dataene viser

**Fase 1 – 2005–2019: Vestlandet systematisk underrepresentert**
I 15 år fikk Vestlandet konsekvent under befolkningsandelen sin i statlig SVV-finansiering.
Initiativ Vest har historisk grunnlag for dette poenget.

**Fase 2 – 2020–2025: Markant opphenting**
E39 Svegatjørn-Rådal (12,2 mrd) og E39 Eiganestunnelen (5,5 mrd) dominerer 2020–22
og gir Vestlandet 52,2 % av all statlig SVV – drevet av to prosjekter.

**Fase 3 – 2026+: Vestlandet dominerer fullstendig**
Med 52,9 mrd under bygging utgjør Vestlandet 50,8 % av fremtidig statlig SVV.
Dette er nesten dobbelt av befolkningsandelen (25,5 %).

**Totalt 2005–2039:** Vestlandet 36,3 % av statlig SVV mot 25,5 % av befolkningen → **+10,8 pp**.

### Det strukturelle poenget om bompengeandel

Lavest statlig andel = Hardangerbrua (10 %), Ryfast (23 %), Rogfast (40 %).
Høyest = rassikring, tunneloppgradering, fjelloverganger (100 %).
Mønsteret er **prosjekttype**, ikke geografi.

### Hordfast-implikasjonen

Med realistisk finansiering (74 mrd kostnad, maks 8 mrd bompenger per Vista-analyse
= **89 % statlig**) ville Hordfast alene flytte Vestlandets SVV-andel fra 36 % til
rundt 48 % – klart over befolkningsandelen. Prosjektet er ikke en oppretting av
historisk skjevhet; opphentingen er allerede i gang.